# 🎯 Module 11: Transfer Learning

> **Stage:** Deep Learning  
> **Level:** Intermediate → Job-Ready  
> **Module:** Transfer Learning  
> **Framework:** PyTorch  
> **Date:** 11 June 2026  
> **Previous:** 10 — Transformers Basics  
> **Next:** 12 — Autoencoders

---

Training deep networks from scratch requires **massive datasets** and **lots of compute**. Most real-world problems don't have that luxury.

**Transfer learning** lets us start from a model that has *already learned useful features* and adapt it to our task with **far less data and time**.

In this notebook we'll cover:

- What transfer learning is and why it works
- **Feature extraction** vs **fine-tuning**
- Freezing and unfreezing layers
- **Discriminative learning rates**
- Using `torchvision.models` with pretrained weights
- Domain similarity and catastrophic forgetting
- A complete image classification project
- Transfer learning for **NLP**

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain **why** pretrained models transfer to new tasks
- Describe the **feature-extraction** vs **fine-tuning** workflows
- Freeze / unfreeze layers with `requires_grad`
- Replace a task-specific head correctly
- Use **discriminative learning rates**
- Load pretrained torchvision models with the correct `weights` API
- Match **preprocessing** to the pretrained model
- Reason about **domain shift** and **catastrophic forgetting**
- Build, train and evaluate a transfer-learning classifier
- Compare **feature extraction**, **fine-tuning**, and **from-scratch** training

---
## ⚙️ 0. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

import torchvision
from torchvision import datasets, transforms, models
from torchvision.models import resnet18, ResNet18_Weights

import numpy as np
import matplotlib.pyplot as plt
import time

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')
print(f'torchvision version: {torchvision.__version__}')

---
## 🧠 1. What Is Transfer Learning?

**Transfer learning** means taking knowledge learned by a model on one task and **reusing it** for another related task.

Instead of starting from **random weights**:

```text
Random weights
    ↓
Train on your small dataset
```

we start from a **pretrained model**:

```text
Pretrained model
      ↓
Reuse learned features
      ↓
Adapt to your task
```

### Benefits

- **Less training time**
- **Less data** required
- **Less compute**
- Often **better generalization** on small datasets

---
## 🎨 2. Why Pretrained Models Work

A CNN trained on millions of images learns **hierarchical features**:

```text
edges
  ↓
corners
  ↓
textures
  ↓
shapes
  ↓
objects
```

The **early layers** learn generic, low-level features that are useful for **almost any** image task.

```text
ImageNet pretrained model
          ↓
Your medical/image dataset
          ↓
New classifier
```

We **reuse** those generic representations instead of learning them from scratch.

---
## 🔄 3. The Transfer Learning Workflow

```text
Pretrained Model
       ↓
Inspect architecture
       ↓
Replace task-specific head
       ↓
Freeze / unfreeze layers
       ↓
Train on new dataset
       ↓
Evaluate
       ↓
Fine-tune if necessary
```

We'll follow this exact recipe throughout the notebook.

---
## 🧊 4. Feature Extraction vs 🔥 Fine-Tuning

### Feature Extraction

Freeze the pretrained backbone, train **only** the new classifier:

```text
Pretrained layers → FROZEN
                    ↓
                New head
                    ↓
                  Train
```

✅ Faster · ✅ Less compute · ✅ Less overfitting risk · ✅ Great for **small datasets**.

### Fine-Tuning

Allow some or all pretrained layers to update:

```text
Pretrained model
       ↓
Unfreeze selected layers
       ↓
Train with small learning rate
```

Lets the model adapt its learned representations to the **new domain** — but requires more data & care.

---
## 📋 5. Common Strategies

| Strategy | Backbone | New Head | Use Case |
| --- | --- | --- | --- |
| **Feature extraction** | ❄️ Frozen | 🔥 Train | Small dataset |
| **Partial fine-tuning** | Partial | 🔥 Train | Moderate adaptation |
| **Full fine-tuning** | 🔥 Trainable | 🔥 Train | Larger / related dataset |
| **Train from scratch** | 🎲 Random | 🔥 Train | Large custom dataset / special case |

### When to freeze?

If your dataset is small (say, 500 images) and the backbone has millions of parameters, training everything immediately will **overfit**.

**Start with feature extraction → evaluate → then fine-tune later layers.**

---
## 🐍 6. Pretrained Models in PyTorch

Modern torchvision API (recommended):

```python
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights=ResNet18_Weights.DEFAULT)
```

The model already contains **pretrained parameters** trained on ImageNet.

Each weights enum also provides the **correct preprocessing** for that model via `weights.transforms()`.

In [ ]:
# Load a pretrained ResNet-18 and inspect it
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

print('Model class:', model.__class__.__name__)
print('Total parameters:', sum(p.numel() for p in model.parameters()), '\n')
print('--- Architecture (top-level modules) ---')
for name, module in model.named_children():
    print(f'  {name:12s} → {module.__class__.__name__}')
print('\nOriginal classifier head:', model.fc)

In [ ]:
# The preprocessing that the pretrained weights expect
preprocess = weights.transforms()
print(preprocess)

---
## ❄️ 7. Freezing the Backbone & Replacing the Head

### Freeze everything

```python
for param in model.parameters():
    param.requires_grad = False
```

Now the pretrained parameters **won't receive gradient updates**.

### Replace the classifier

For a 3-class problem (`cat`, `dog`, `horse`):

```python
model.fc = nn.Linear(model.fc.in_features, 3)
```

The new head remains **trainable** by default.

In [ ]:
# Freeze + replace head, then count trainable params
model = resnet18(weights=ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

# Replace final classifier with a new 3-class head
num_classes = 3
model.fc = nn.Linear(model.fc.in_features, num_classes)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}')
print(f'\nOnly {100 * trainable / total:.2f}% of parameters are being trained.')

In [ ]:
# Forward pass: verify the model works with the new head
model = model.to(device)
x = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    y = model(x)
print('Input :', tuple(x.shape))
print('Output:', tuple(y.shape), ' → [batch, num_classes]')

---
## 📦 8. Dataset & Preprocessing

We'll use **CIFAR-10** as a stand-in for a real project. Two important notes:

1. The pretrained ResNet expects **224×224** input with **ImageNet normalization**.
2. CIFAR-10 images are 32×32 — so we **resize** them up.

This mismatch is realistic and mirrors what happens when you apply an ImageNet backbone to a new dataset.

In [ ]:
# Build transforms: train (with augmentation) and val (no augmentation)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Load a subset of CIFAR-10 (smaller = faster demo)
full_train = datasets.CIFAR10(root='./data', train=True,  download=True, transform=train_tf)
test_ds    = datasets.CIFAR10(root='./data', train=False, download=True, transform=eval_tf)

# Subsample train and carve out a validation split
torch.manual_seed(42)
sub_idx = torch.randperm(len(full_train))[:6000].tolist()
train_subset = torch.utils.data.Subset(full_train, sub_idx)

n_val = 1000
n_tr  = len(train_subset) - n_val
train_ds, val_ds = random_split(
    train_subset, [n_tr, n_val],
    generator=torch.Generator().manual_seed(42),
)

BATCH = 64
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2)

CLASSES = full_train.classes
print('Train:', len(train_ds), ' Val:', len(val_ds), ' Test:', len(test_ds))
print('Classes:', CLASSES)

In [ ]:
# Show a few sample images
def denorm(img):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (img * std + mean).clamp(0, 1)

imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for ax, img, lbl in zip(axes, imgs[:8], labels[:8]):
    ax.imshow(denorm(img).permute(1, 2, 0))
    ax.set_title(CLASSES[lbl], fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

---
## 🏋️ 9. Training & Evaluation Helpers

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total

def run_training(model, optimizer, epochs, name):
    criterion = nn.CrossEntropyLoss()
    hist = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_acc = evaluate(model, val_loader, criterion)
        hist['train_loss'].append(tr_loss); hist['val_loss'].append(va_loss)
        hist['train_acc'].append(tr_acc);   hist['val_acc'].append(va_acc)
        print(f'[{name}] Epoch {epoch:2d}/{epochs} | '
              f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f} | '
              f'val_loss={va_loss:.4f}  val_acc={va_acc:.4f}')
    dt = time.time() - t0
    print(f'[{name}] Done in {dt:.1f}s | best val_acc={max(hist["val_acc"]):.4f}')
    return hist, dt

print('Helpers ready.')

---
## 🅰️ 10. Model A — Feature Extraction

Backbone **frozen**, only the new head is trained.

```python
optimizer = torch.optim.AdamW(model.fc.parameters(), lr=1e-3)
```

In [ ]:
torch.manual_seed(42)

model_A = resnet18(weights=ResNet18_Weights.DEFAULT)
for p in model_A.parameters():
    p.requires_grad = False
model_A.fc = nn.Linear(model_A.fc.in_features, len(CLASSES))
model_A = model_A.to(device)

opt_A = optim.AdamW(model_A.fc.parameters(), lr=1e-3)

hist_A, time_A = run_training(model_A, opt_A, epochs=5, name='FeatureExtract')

---
## 🔥 11. Model B — Fine-Tuning with Discriminative Learning Rates

Unfreeze the **last residual block** (`layer4`) and train it with a **smaller learning rate** than the new head.

```python
optimizer = torch.optim.AdamW([
    {"params": model.layer4.parameters(), "lr": 1e-5},
    {"params": model.fc.parameters(),     "lr": 1e-3},
])
```

### Why smaller LR for pretrained layers?

```text
Pretrained weights
      ↓
small updates
      ↓
adapt to new task
```

rather than:

```text
Pretrained weights
      ↓
huge updates
      ↓
forget useful representations  ← catastrophic forgetting
```

In [ ]:
torch.manual_seed(42)

model_B = resnet18(weights=ResNet18_Weights.DEFAULT)
# Freeze everything
for p in model_B.parameters():
    p.requires_grad = False
# Replace head
model_B.fc = nn.Linear(model_B.fc.in_features, len(CLASSES))
# Unfreeze last residual block
for p in model_B.layer4.parameters():
    p.requires_grad = True
model_B = model_B.to(device)

trainable_B = sum(p.numel() for p in model_B.parameters() if p.requires_grad)
print(f'Trainable params in Model B: {trainable_B:,}  '
      f'({100 * trainable_B / sum(p.numel() for p in model_B.parameters()):.2f}% of total)\n')

# Discriminative learning rates
opt_B = optim.AdamW([
    {'params': model_B.layer4.parameters(), 'lr': 1e-5},
    {'params': model_B.fc.parameters(),     'lr': 1e-3},
])

hist_B, time_B = run_training(model_B, opt_B, epochs=5, name='FineTune')

---
## 🅲 12. Model C — Training From Scratch

The same architecture with **random weights** — no transfer learning at all.

This is the baseline for comparison.

In [ ]:
torch.manual_seed(42)

# resnet18(weights=None) → random init
model_C = resnet18(weights=None)
model_C.fc = nn.Linear(model_C.fc.in_features, len(CLASSES))
model_C = model_C.to(device)

opt_C = optim.AdamW(model_C.parameters(), lr=1e-3)

hist_C, time_C = run_training(model_C, opt_C, epochs=5, name='Scratch')

---
## 📊 13. Comparing the Three Strategies

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for hist, label in [(hist_A, 'A: Feature extraction'),
                    (hist_B, 'B: Fine-tuning'),
                    (hist_C, 'C: From scratch')]:
    axes[0].plot(hist['val_loss'], label=label)
    axes[1].plot(hist['val_acc'],  label=label)
axes[0].set_title('Validation loss');     axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title('Validation accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Feature Extraction vs Fine-Tuning vs From-Scratch')
plt.tight_layout(); plt.show()

In [ ]:
# Summary table
def trainable_count(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

rows = [
    ('A: Feature extraction', trainable_count(model_A), time_A, hist_A['val_acc'][-1]),
    ('B: Fine-tuning',        trainable_count(model_B), time_B, hist_B['val_acc'][-1]),
    ('C: From scratch',       trainable_count(model_C), time_C, hist_C['val_acc'][-1]),
]

print(f'{"Strategy":<24} {"Trainable":>12} {"Time (s)":>10} {"Best Val Acc":>14}')
print('-' * 64)
for name, tr, t, acc in rows:
    print(f'{name:<24} {tr:>12,} {t:>10.1f} {acc:>14.4f}')

print('\nTakeaways:')
print('  - Feature extraction trains a tiny fraction of params → fastest.')
print('  - Fine-tuning usually gives the best accuracy on modest datasets.')
print('  - From scratch needs MUCH more data + time to match pretrained models.')

---
## 🌍 14. Domain Similarity Matters

Transfer learning works best when the **source** and **target** tasks share useful structure.

```text
General image classification
        ↓
Animal classification       ← works well
```

```text
Natural images
        ↓
Specialized scientific images   ← often needs heavier fine-tuning
```

The farther the target domain is from the pretraining data, the more **careful** you must be.

### What to do when domains differ a lot

- Unfreeze **more** layers
- Use **longer** fine-tuning
- Consider a **domain-appropriate** backbone
- Consider **training from scratch** if you have enough data

---
## 💬 15. Transfer Learning for NLP

The same idea applies to text. A **pretrained Transformer** already learned rich language representations:

```text
Pretrained Transformer (BERT, RoBERTa, GPT, …)
        ↓
Text classification
```

Typical adaptation targets:

- Sentiment classification
- Spam detection
- Topic classification
- Named entity recognition

Typical recipe (using HuggingFace `transformers`):

```python
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2
)
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
```

The same conceptual workflow applies: **replace the head, optionally freeze, train with small LR, evaluate.**

---
## 🚨 16. Common Mistakes

### ❌ 1. Training everything immediately
On a tiny dataset, this overfits fast. **Start with feature extraction.**

### ❌ 2. Using a large learning rate on pretrained layers
Pretrained weights get destroyed. Use **smaller LR** for pretrained parts.

### ❌ 3. Forgetting `requires_grad`
Frozen parameters should have:
```python
param.requires_grad = False
```

### ❌ 4. Wrong preprocessing
Input normalization and resize **must match** the pretrained model's expected setup:
```python
preprocess = weights.transforms()   # use the enum's transforms
```

### ❌ 5. Data leakage
Never allow validation/test info to influence training (including through augmentation stats or hyperparameter tuning on the test set).

### ❌ 6. Tuning on the test set
Use **validation** for decisions; touch **test** only once, at the end.

---
## 🏋️ 17. Hands-On Exercises

### Exercise 1 — Try ResNet-50
Swap `resnet18` for `resnet50`. Compare accuracy vs training time.

### Exercise 2 — Unfreeze More Layers
In Model B, also unfreeze `layer3`. Does accuracy improve?

### Exercise 3 — Discriminative LR Sweep
Try `lr_backbone ∈ {1e-6, 1e-5, 1e-4}` and plot val accuracy.

### Exercise 4 — Freeze vs Full Fine-Tune
Compare **feature extraction**, **partial fine-tune**, and **full fine-tune**. Which wins on CIFAR-10 subset?

### Exercise 5 — Different Backbone
Try `efficientnet_b0` or `mobilenet_v3_small`. Compare parameter count and accuracy.

### Exercise 6 — Medical / Small Domain
Simulate **domain shift**: train on a heavily blurred CIFAR-10, evaluate on the original. How much does fine-tuning help?

### Exercise 7 — NLP
Use HuggingFace `transformers` to fine-tune `distilbert-base-uncased` on a small sentiment dataset (IMDb subset, SST-2).

---
## 🎤 18. Interview Questions

1. **What is transfer learning?**  
   Reusing knowledge from a pretrained model for a new related task.

2. **What is feature extraction?**  
   Using a pretrained model as a mostly **frozen feature generator** while training a new task-specific head.

3. **What is fine-tuning?**  
   Updating some or all pretrained parameters on the target dataset.

4. **Why use a smaller learning rate for pretrained layers?**  
   To adapt existing representations without destroying them (avoid catastrophic forgetting).

5. **Why freeze layers?**  
   To reduce computation, train fewer parameters, and reduce overfitting risk.

6. **What is domain shift?**  
   A difference between the data distribution used for pretraining and the target data distribution.

7. **What is catastrophic forgetting?**  
   When fine-tuning with too large updates causes the model to lose useful pretrained representations.

8. **Which layers transfer best?**  
   **Early layers** (edges, textures) transfer broadly; **later layers** are more task-specific.

9. **When would you NOT use transfer learning?**  
   When the target domain is very different, or when you have a massive labeled dataset and enough compute to train from scratch.

10. **What must match between pretraining and your training?**  
    Preprocessing — resize, normalization mean/std, and input channel layout.

---
## ✅ 19. Knowledge Check

- [ ] Explain what transfer learning is
- [ ] Explain why pretrained models transfer
- [ ] Describe the transfer-learning workflow
- [ ] Distinguish **feature extraction** vs **fine-tuning**
- [ ] Freeze parameters with `requires_grad`
- [ ] Replace a classifier head correctly
- [ ] Explain **discriminative learning rates**
- [ ] Load pretrained models with `weights=...DEFAULT`
- [ ] Use `weights.transforms()` for preprocessing
- [ ] Explain **domain shift** and **catastrophic forgetting**
- [ ] Apply transfer learning to NLP
- [ ] Compare feature-extraction vs fine-tuning vs from-scratch

---
## 🏁 20. Key Takeaways

### The workflow

```text
Pretrained Model
       ↓
Reuse learned representations
       ↓
Replace task-specific head
       ↓
Freeze OR fine-tune
       ↓
Evaluate
```

### Two core strategies

> **Feature extraction** = reuse the model mostly as-is.  
> **Fine-tuning** = adapt pretrained weights to your task.

### The recommended progression

```text
Start frozen
     ↓
Train new head
     ↓
Evaluate
     ↓
Unfreeze selected layers
     ↓
Fine-tune with smaller LR
```

### 🧭 Golden Rule

> **Start frozen, then unfreeze gradually with smaller learning rates. Match preprocessing exactly, and always evaluate on a proper validation set.**

---

### 🔗 What's Next?

**Module 12 — Autoencoders**

We move from **reusing learned representations** to **learning compressed representations of data** through an encoder and decoder — the foundation of modern generative models.